In [ ]:
%load_ext autoreload
%autoreload 2
%reset -f

In [ ]:
import sys
import os
import logging
from pathlib import Path

from locallib.picarrodb import *
from locallib.box import *
from locallib.pandas import *
from locallib.query import *

from datetime import datetime
from slack_bolt import App

# Get the project root directory correctly (one level up from extractor/)
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
if not os.path.isdir(os.path.join(project_root, "locallib")):
    project_root = notebook_dir

# Add the lib directory to sys.path for lib package imports
if project_root not in sys.path:
    sys.path.insert(0, project_root)

LOG_DIR = os.path.join(project_root, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, "extractor.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.FileHandler(LOG_FILE, mode="a")],
    force=True,
)
logger = logging.getLogger("extractor")


class _TeeStream:
    def __init__(self, stream):
        self.stream = stream
        self._buffer = ""

    def write(self, data):
        if not data:
            return 0
        self.stream.write(data)
        self._buffer += data
        while "\n" in self._buffer:
            line, self._buffer = self._buffer.split("\n", 1)
            if line:
                logger.info(line)
        return len(data)

    def flush(self):
        self.stream.flush()
        if self._buffer:
            logger.info(self._buffer)
            self._buffer = ""


sys.stdout = _TeeStream(sys.__stdout__)
print(f"Logging to {LOG_FILE}")

import lib.custom_pandas
from lib.query import *
from lib.input_output import *
from lib.msapi import *
from lib.config import *
from lib.tables.IngesterTables import *
from lib.KPIHubConnection import *

# Configuration section

In [ ]:
#Get active customers
customers = PeakAboveSATCustomer.query_table(arguments={'db_path': DB_PATH})
start_date = STARTING_DATE.strftime('%Y-%m-%d')
end_date = datetime.now().strftime('%Y-%m-%d')
#Send flags
send_email = True

#Set up bot
SLACK_BOT_TOKEN = os.getenv("SLACKBOTTOKEN")
app = App(token=SLACK_BOT_TOKEN)

#Output Columns
OUT_COLS=['CustomerName','PeakName','PeakId','Date','WeekNumber','Disposition','LocalTime','BoundaryName','Region','SubRegion','Plant','EmissionRate','PeakGpsLatitude','PeakGpsLongitude','Easting','Northing','UserName','SurveyorUnit','AnalyzerSerialNumber','Hyperlink','LastUpdated']
DB_COLS=['CustomerId','PeakName','PeakId','Date','WeekNumber','Disposition','LocalTime','BoundaryName','Region','SubRegion','Plant','EmissionRate','PeakGpsLatitude','PeakGpsLongitude','Easting','Northing','UserName','SurveyorUnit','AnalyzerSerialNumber','Hyperlink','LastUpdated']

# Loop for every customer in the list

In [ ]:
for _,customer_info in customers.iterrows():
    customer_name = customer_info['Name']
    print(f'-----{customer_name}')
    customer_id = customer_info['CustomerId']
    threshold = float(customer_info['ThresholdSCFH'])
    connection = CONN_DICT[customer_info['DBLocation']]
    box_folder_id = customer_info['BoxFolderId']
    xchange_location = customer_info['XchangeLocation']

    #Generate the file names
    daily_filename = f'{customer_name}_{SUFFIX}_{datetime.now().strftime("%Y-%m-%d")}.xlsx'
    filename = f'{customer_name}_{SUFFIX}.xlsx'
    #Get the recipients
    recipients = PeakAboveSATRecipients.query_table(arguments={'db_path': DB_PATH})
    recipients = recipients[(recipients['Active'] == True) & (recipients['CustomerId'] == customer_id)]

    # --- Query data from P-Cubed
    output_log = "="*20+'\n'
    output_log = f'Starting the process for {customer_name}  \n '
    output_log += f'Process started at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
    #Add current date to the output log as a header
    output_log += "="*20+'\n'
    #Query the data from the EU2 Database
    a = get_users(customer_name, '#UserList')
    b = get_surveys('#UserList', '#SurveyList',start_date,end_date)
    b.set_child(get_peak_table('#SurveyList','#UserList',threshold))
    a.set_child(b)
    current_data = a.execute(connection)
    db_records = len(current_data)
    print('Number of peaks in the P-Cubed Database:',db_records)

    # --- Retrieve the data from the local base
    local_data = Query(f"SELECT * FROM PeakAboveSAT WHERE CustomerId =  '{customer_id}'").execute(KPIHub_Conn)

    # --- Check the number of peaks
    if len(current_data) > 0:
        #Check the peaks that are not in the file
        if local_data is not None:
            process_data = current_data[~current_data['PeakName'].isin(local_data['PeakName'])]
        else:
            #No file found, make the file for the first time
            process_data = current_data
        new_peaks = len(process_data)
        print('There are',len(process_data),'new peaks')
        output_log += f'There are {len(process_data)} new peaks\n'
    else:
        print('There are no SAT peaks between',start_date,'and',end_date)
        process_data = pd.DataFrame()

    # --- Process the data
    if len(process_data) > 0:
        #Set esting and northing
        process_data.DA3540.add_easting_northing()

        #Change epoch start to UK date and local time
        process_data.DA3540.epoch_to_local_time()

        #Process the region
        boundaries = pd.DataFrame(process_data['BoundaryName'].unique(), columns=['BoundaryName'])
        query = f"SELECT externalid as BoundaryName, region as Region, subregion as SubRegion, plant as plant FROM xchange.{xchange_location} WHERE externalid IN (SELECT BoundaryName FROM temp_boundaries)"
        boundaries.db.set_query(query)
        regions =boundaries.db.execute(DATAHUB_Conn, temp_table_name = 'temp_boundaries', source_col = 'BoundaryName')
        process_data['Region'] = process_data['BoundaryName'].map(regions.set_index('boundaryname')['region'])
        process_data['SubRegion'] = process_data['BoundaryName'].map(regions.set_index('boundaryname')['subregion'])
        process_data['Plant'] = process_data['BoundaryName'].map(regions.set_index('boundaryname')['plant'])
        process_data['Date'] = pd.to_datetime(process_data['Date'], errors='coerce')
        process_data['WeekNumber'] = process_data['Date'].dt.isocalendar().week
        process_data['LastUpdated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        #Process the week number
        process_data['WeekNumber'] = process_data['Date'].dt.isocalendar().week
        
        daily_report = process_data.copy()

        #Add hyperlink column using SurveyId
        daily_report['Hyperlink'] = daily_report['SurveyId'].apply(lambda x: f"https://pcubed2.eu.picarro.com/Live/Survey/{x.lower()}" if pd.notna(x) else "")

        #Order by date and then local time
        daily_report.sort_values(by=['Date','LocalTime'], inplace=True, ascending=False)
        daily_report['CustomerId'] = customer_id

        PeakAboveSAT.update_table(arguments={'db_path': DB_PATH , 'DataFrame': daily_report[DB_COLS], 'PrimaryKey': 'PeakId'})

    # --- Sanity check
    local_data = Query(f"SELECT * FROM PeakAboveSAT WHERE CustomerId =  '{customer_id}'").execute(KPIHub_Conn)
    num_local_data = len(local_data)
    output_log += f'Number of peaks in the local database: {num_local_data}\n'
    print('Number of peaks in the local database:',num_local_data)
    output_log += f'Number of peaks in the P-Cubed database: {db_records}\n'
    print('Number of peaks in the P-Cubed database:',db_records)
    output_log += "="*20+'\n'

    # --- Upload the general file
    if (len(process_data) > 0):
        #--- Upload the general file
        # Get all the data
        local_data.drop(columns = ['CustomerId'], inplace = True)
        local_data['CustomerName'] = customer_name
        local_data.sort_values(by=['Date','LocalTime'], inplace=True, ascending=False)    
        local_data[OUT_COLS].to_excel(filename, index=False)
        auto_adjust_excel_columns(filename)

        #Generate the file objecs to upload the data
        current_daily_folder_name = f'{end_date}'
        daily_report_name = f'{customer_name}_{SUFFIX}_{end_date}.xlsx'
        box_file = BoxFile(filename,str(box_folder_id))
        box_file.upload()


    # --- Send the email
    if send_email and (len(process_data) > 0):
        #Set the accumulative data via email
        for rcpt in recipients['Email']:
                try:
                    send_email_via_outlook_api(
                        subject=f'{customer_name} - Peaks above SAT - New {new_peaks} peaks found',
                        body=f'Generated on {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}.',
                        recipient=rcpt,
                        attachments=[os.path.join(notebook_dir, filename)]
                    )
                    print(f"✅ Email sent successfully to {rcpt}")
                    #box_file.delete()

                except Exception as e:
                    print(f"❌ Failed to send email to {rcpt}: {str(e)}")
    else:
        print('No new peaks to process')
        #box_file.delete()
        output_log += 'No new peaks to process\n'


    